# ST-Reason · 05 LLM question answering

Qwen3-8B (4-bit) answers every question from the memories built in notebook 04. One **run** is one combination of question set × memory format × strategy. The runs are the Tier 1 experiments of the plan:

| Experiment | Question set | Memories | Strategies |
| --- | --- | --- | --- |
| **E3** memory ablation (H1) | main grid (400) and STAR (360) | CAP, M1, M2, M3, M4, VA | A, D |
| **E4** temporal gap (H2) | STAR gap questions (120) × pause 10/25 s and sham 10/25 s | M1, M4 | D |
| **Oracle** (H4) | STAR (360) | ORACLE_M1, ORACLE_M4 | A, D |
| **E5** strategies (H3) | main grid (400) | M4 | B, C |
| **Controls** on dev (pilot rule) | dev (300) | M4 | A, D, A-SC, D-X |

| Strategy | What it does | LLM calls |
| --- | --- | --- |
| A Direct | memory + question + options → answer + cited item IDs | 1 |
| B Decomposition | parse into a schema and 2–4 sub-questions → answer them in order → final answer | 2 |
| C Retrieval | parse → retrieve memory items (BM25 + bge-small, 600-token budget) → answer from those items only | 2 |
| D Retrieval + verification | C, then a verifier checks support; if not supported, retrieve with twice the budget and answer once more | 3–4 |
| A-SC | A sampled 3 times at temperature 0.7, majority vote (compute control) | 3 |
| D-X | D with a different model as verifier (Gemma 3 4B if a Hugging Face token is set, otherwise Qwen3-4B) | 3–4 |

**Four people, in parallel.** Each person runs their own file (`SHARD` 0–3), which answers a quarter of the questions in **every** run, so the four parts are equal and independent. Everyone can start whenever they like.

**Which runs.** Shard 0 answers every core run (about 7 GPU hours over 4 sessions): M1 vs M4 under D (H1), D vs A and A-SC (H3), the pause/sham runs (H2), the oracle runs (H4), and the CAP and VA baselines. Shards 1–3 answer only the headline runs (`LITE`, about 2.8 GPU hours, 2 sessions): main and STAR M4/M1 under D, main M4 under A, and the 10 s pause and sham runs under M4. Set `CORE_ONLY = False` later for the optional extras; answered questions are always skipped.

**Saving.** Answers are written to the Colab disk and uploaded to `outputs/predictions/` as verified zips every few minutes. After a disconnect, Run all again: answered questions are skipped. Each run of the main cell stops cleanly after 110 minutes; run that cell again to continue.

**Runtime:** GPU (T4).

In [1]:
# --- Parameters: the only cell to edit -----------------------------------------------------------
SHARD = 1                 # person 1: 0, person 2: 1, person 3: 2, person 4: 3
N_SHARDS = 4
ONLY_RUNS = ['main.M1.A', 'star.M1.A']   # e.g. ['main.M4.D'] to run just some runs; None = all, in priority order
LIMIT_QUESTIONS = None    # e.g. 8 for a quick test (per run); None for the real run
SYNC_EVERY_MIN = 8        # upload answers to Drive at least this often (minutes)
CORE_ONLY = False         # True: only the runs the main tests need. False: also the optional extras
LITE = (SHARD != 0)       # shards 1-3: only the headline runs (~2.8 h). Shard 0 does every core run (~7 h)

In [2]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '05_llm_qa'
ST_QUIET = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'
assert SHARD in range(N_SHARDS)
MANIFEST_PATH = P['runs'] / f'manifest_shard{SHARD}.csv'
DRIVE_LOG_PATH = LOG_PATH                                   # log locally, copy to Drive with each upload
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
import pandas as pd, numpy as np

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87


## 1. Memories and questions

The newest memory zip from notebook 04 is unpacked to the Colab disk. Questions come from the frozen sets. Each question belongs to one shard, decided by a fixed hash of its ID, so the split is the same on every computer.

In [3]:
MEM_LOCAL = P['local'] / 'memory_in'
packs = sorted(P['memory'].glob('04_memories_*.zip'))
assert packs, 'no memory zip in outputs/memory: run notebook 04 first'
MEM_PACK = packs[-1].name
if load_json(MEM_LOCAL / 'source.json', default={}).get('pack') != MEM_PACK:
    shutil.rmtree(MEM_LOCAL, ignore_errors=True)
    MEM_LOCAL.mkdir(parents=True)
    local_zip = P['local_tmp'] / MEM_PACK
    shutil.copyfile(packs[-1], local_zip)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(MEM_LOCAL)
    save_json({'pack': MEM_PACK}, MEM_LOCAL / 'source.json')
print('memories from', MEM_PACK, '| formats:', sorted(p.name for p in MEM_LOCAL.iterdir() if p.is_dir()))

_MEM = {}
def memory(fmt, key):
    k = (fmt, key)
    if k not in _MEM:
        _MEM[k] = load_json(MEM_LOCAL / fmt / f'{key}.json')
    return _MEM[k]

def read_set(name):
    return pd.read_csv(P['subsets'] / f'{name}.csv', dtype={'video_id': str})
def shard_of(uid):
    return int(hashlib.sha1(str(uid).encode()).hexdigest(), 16) % N_SHARDS

def nextqa_questions(df):
    return [{'uid': r.uid, 'key': str(r.video_id), 'question': r.question.rstrip('?') + '?',
             'options': [str(getattr(r, f'a{i}')) for i in range(5)], 'gold': int(r.answer_idx),
             'type': r.type} for r in df.itertuples(index=False)]
def star_questions(df, version='orig'):
    out = []
    for r in df.itertuples(index=False):
        n = int(getattr(r, 'n_options', 4) or 4)
        out.append({'uid': r.uid if version == 'orig' else f'{r.uid}__{version}', 'base_uid': r.uid,
                    'key': f'{r.uid}__{version}', 'question': str(r.question),
                    'options': [str(getattr(r, f'a{i}')).rstrip('.') for i in range(n)],
                    'gold': int(r.answer_idx), 'type': r.type, 'version': version})
    return out

QSETS = {'main': nextqa_questions(read_set('main_grid_eval')),
         'dev': nextqa_questions(read_set('dev')),
         'star': star_questions(read_set('star_main'))}
gap = read_set('star_gap')
for v in ['pause10', 'sham10', 'pause25', 'sham25']:
    QSETS[f'gap_{v}'] = star_questions(gap, v)
for name, qs in QSETS.items():
    for q in qs:
        q['shard'] = shard_of(q.get('base_uid', q['uid']))
print({n: f"{sum(q['shard'] == SHARD for q in qs)} of {len(qs)}" for n, qs in QSETS.items()})

memories from 04_memories_20261017-124338.zip | formats: ['CAP', 'M1', 'M2', 'M3', 'M4', 'ORACLE_M1', 'ORACLE_M2', 'ORACLE_M3', 'ORACLE_M4', 'VA']
{'main': '93 of 400', 'dev': '83 of 300', 'star': '96 of 360', 'gap_pause10': '29 of 120', 'gap_sham10': '29 of 120', 'gap_pause25': '29 of 120', 'gap_sham25': '29 of 120'}


## 2. The runs, in priority order

In [4]:
def R(qset, fmt, strat):
    return {'id': f'{qset}.{fmt}.{strat}', 'qset': qset, 'format': fmt, 'strategy': strat}
CORE = [
    # H1 (M1 vs M4 under D) and H3 (D vs A, D vs A-SC)
    R('main', 'M4', 'D'), R('main', 'M1', 'D'), R('main', 'M4', 'A'), R('star', 'M4', 'D'), R('star', 'M1', 'D'),
    R('dev', 'M4', 'D'), R('dev', 'M4', 'A'), R('dev', 'M4', 'A-SC'),
    # H2 (pause vs sham)
    *[R(f'gap_{v}', f, 'D') for v in ['pause10', 'sham10', 'pause25', 'sham25'] for f in ['M4', 'M1']],
    # H4 (perception vs reasoning)
    R('star', 'ORACLE_M4', 'D'), R('star', 'ORACLE_M1', 'D'),
    # baselines (one call each)
    *[R(s_, f, 'A') for s_ in ['main', 'star'] for f in ['CAP', 'VA']],
]
EXTRA = [
    R('main', 'M1', 'A'), R('star', 'M4', 'A'), R('star', 'M1', 'A'),
    *[R(s_, f, st) for s_ in ['main', 'star'] for f in ['M2', 'M3'] for st in ['D', 'A']],
    *[R(s_, f, 'D') for s_ in ['main', 'star'] for f in ['CAP', 'VA']],
    *[R('star', f, 'A') for f in ['ORACLE_M4', 'ORACLE_M1']],
    R('main', 'M4', 'B'), R('main', 'M4', 'C'), R('dev', 'M4', 'D-X'),
]
# LITE: the headline comparisons get all 4 shards (400 main, 360 STAR questions); the gap 10 s runs get 128 per version.
# Everything else (dev, 25 s gaps, M1 gaps, oracle, CAP/VA baselines) is answered by shard 0 only.
LITE_IDS = ['main.M4.D', 'main.M1.D', 'main.M4.A', 'star.M4.D', 'star.M1.D', 'gap_pause10.M4.D', 'gap_sham10.M4.D']
RUNS = CORE + ([] if CORE_ONLY else EXTRA)
if LITE and not ONLY_RUNS:
    RUNS = [r for r in RUNS if r['id'] in set(LITE_IDS)]
if ONLY_RUNS:
    RUNS = [r for r in RUNS if r['id'] in set(ONLY_RUNS)]
print(len(RUNS), 'runs:', ', '.join(r['id'] for r in RUNS))

2 runs: main.M1.A, star.M1.A


## 3. Saving answers (verified zips)

Answers go to `pred/<run>/` on the Colab disk, in small files that are never changed once written. Every few minutes the new files are zipped, copied to `outputs/predictions/`, and read back from Drive. At the start, all zips (from every person) are unpacked, so finished questions are skipped and the summary covers everyone's answers so far.

In [5]:
LOCAL_PRED = P['local'] / 'pred_out'
LOCAL_PRED.mkdir(parents=True, exist_ok=True)
PRED_DRIVE = P['pred']
_PACKED = set()

class DriveSyncError(BaseException):
    pass

def restore_preds():
    n = 0
    for z in sorted(PRED_DRIVE.glob('05_shard*.zip'), reverse=True):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    _PACKED.add(m)
                    out = LOCAL_PRED / m
                    if not out.exists():
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(m) as s, open(out, 'wb') as d:
                            shutil.copyfileobj(s, d)
                        n += 1
        except (zipfile.BadZipFile, OSError):
            log(f'damaged zip skipped: {z.name}', 'WARN')
    log(f'restored {n} answer files from Drive')

def sync_preds():
    new = [str(p.relative_to(LOCAL_PRED)) for p in LOCAL_PRED.rglob('*.jsonl') if str(p.relative_to(LOCAL_PRED)) not in _PACKED]
    if not new:
        return 0
    name = f'05_shard{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S}_{len(new):05d}.zip'
    tmp = P['local_tmp'] / name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for rel in new:
            zf.write(LOCAL_PRED / rel, rel)
    ok = False
    for attempt in range(3):
        try:
            shutil.copyfile(tmp, PRED_DRIVE / name)
            with zipfile.ZipFile(PRED_DRIVE / name) as zf:
                ok = len(zf.namelist()) == len(new) and zf.testzip() is None
            if ok:
                break
        except Exception as e:
            log(f'upload of {name} failed ({e.__class__.__name__}); retrying', 'WARN')
        time.sleep(5 * (attempt + 1))
    if not ok:
        raise DriveSyncError(f'Drive did not keep {name}. Answers are still on this Colab disk: do NOT restart; '
                             'run the summary cell (section 7) to retry the upload.')
    _PACKED.update(new)
    tmp.unlink(missing_ok=True)
    try:
        shutil.copyfile(LOG_PATH, DRIVE_LOG_PATH)
    except Exception:
        pass
    log(f'uploaded {name}: {len(new)} files, verified on Drive')
    return len(new)

def answered(run_id):
    out = set()
    for f in (LOCAL_PRED / run_id).glob('*.jsonl'):
        out |= {r['uid'] for r in read_jsonl(f)}
    return out

restore_preds()

[05:08:41] INFO  restored 63 answer files from Drive


## 4. Models

Qwen3-8B in 4-bit NF4 is the reasoner (thinking mode off). bge-small embeds memory items for retrieval. The D-X verifier is loaded only when that run starts. Answers are capped at short lengths (the pilot rule: 64 tokens for a plain answer).

In [6]:
if not MOCK:
    pip_install('transformers>=4.57.0', 'accelerate>=1.0', 'bitsandbytes>=0.45', 'rank_bm25')
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
HAS_TOKEN = hf_login() if not MOCK else False

def load_llm(spec):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    kw = {'device_map': {'': 0}, 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main')}
    if spec.get('quant') == 'nf4':
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kw['dtype'] = torch.float16
    tok = AutoTokenizer.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    try:
        model = AutoModelForCausalLM.from_pretrained(spec['id'], **kw)
    except (ValueError, KeyError):                       # Gemma 3 4B is multimodal
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    devs = {str(p.device) for p in model.parameters()}
    if any(not d.startswith('cuda') for d in devs):
        raise RuntimeError(f'model not fully on the GPU ({devs}): Disconnect and delete runtime, then Run all once.')
    return model.eval(), tok

_GEN_CAP = {}   # learned batch size per (model, task length): survives between calls, so it does not run out of memory again and again
def llm_generate(model, tok, convs, max_new_tokens, temperature=0.0, batch_size=8, max_batch_tokens=7000):
    """Batched chat generation (thinking off), token-budgeted, with OOM backoff. Returns (texts, prompt_lens, gen_lens)."""
    import torch, gc
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in convs]
    lens = [len(tok(t)['input_ids']) for t in texts]
    order = sorted(range(len(texts)), key=lambda k: lens[k])
    outs, glens = [None] * len(texts), [0] * len(texts)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    gen.update(dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False))
    key = (id(model), max_new_tokens)
    cap = min(batch_size, _GEN_CAP.get(key, batch_size))
    i, ok_streak = 0, 0
    while i < len(order):
        j = i + 1
        while j < len(order) and j - i < cap and (j - i + 1) * (lens[order[j]] + max_new_tokens) <= max_batch_tokens:
            j += 1
        idx = order[i:j]
        oom = False
        try:
            enc = tok([texts[k] for k in idx], return_tensors='pt', padding=True).to(model.device)
            with torch.inference_mode():
                out = model.generate(**enc, **gen)
            new = out[:, enc['input_ids'].shape[1]:]
            for k, row, d in zip(idx, new, tok.batch_decode(new, skip_special_tokens=True)):
                outs[k], glens[k] = d, int((row != tok.pad_token_id).sum())
            del enc, out, new
            i = j
            if len(idx) > _GEN_CAP.get(key, batch_size):    # a bigger batch worked: remember it
                _GEN_CAP[key] = len(idx)
            ok_streak += 1
            if ok_streak >= 4 and cap < batch_size:        # try one bigger again after 4 good batches
                cap += 1; ok_streak = 0
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:                                           # free memory OUTSIDE the except block, or it stays held
            enc = out = new = None
            gc.collect(); torch.cuda.empty_cache()
            if len(idx) == 1:
                raise torch.cuda.OutOfMemoryError('one prompt alone does not fit on the GPU')
            cap = max(1, len(idx) // 2); ok_streak = 0
            _GEN_CAP[key] = cap
            log(f'GPU memory full at batch {len(idx)}: retrying with batch {cap}', 'WARN')
    return outs, lens, glens

if MOCK:
    class _T:
        pad_token_id = 0
        def __call__(self, t):
            return {'input_ids': t.split()}
        def apply_chat_template(self, m, **k):
            return ' '.join(x['content'] for x in m)
    def load_llm(spec):
        return None, _T()
    def llm_generate(model, tok, convs, max_new_tokens, temperature=0.0, batch_size=8, max_batch_tokens=7000):
        outs = []
        for m in convs:
            u = m[-1]['content']
            ids = re.findall(r'\b(E\d+|P\d+|C\d+)\b', m[-2]['content'] if len(m) > 2 else u) or ['E1']
            if 'supported' in u:
                outs.append('{"supported": %s}' % ('true' if len(u) % 3 else 'false'))
            elif 'sub_questions' in u:
                outs.append('{"type": "temporal", "target_object": "cup", "target_actor": "person", "anchor_event": "pick up", '
                            '"relation": "after", "sub_questions": ["What happens first?", "What happens next?"]}')
            else:
                letters = re.findall(r'^([A-E])\. ', u, flags=re.M) or ['A']
                outs.append('{"answer": "%s", "evidence": ["%s"], "reasoning_steps": ["checked"]}' % (letters[len(u) % len(letters)], ids[0]))
        return outs, [len(' '.join(x['content'] for x in m).split()) for m in convs], [20] * len(convs)

MAIN_SPEC = CFG['models']['reasoner_main']
if 'LLM' in globals() and LLM is not None:
    print('reasoner already loaded in this runtime: reusing it')       # Run all again without a restart is fine
else:
    if not MOCK:
        import torch
        free_memory()
        if torch.cuda.memory_allocated() / 1e9 > 1.0:
            raise RuntimeError('GPU memory already in use: Disconnect and delete runtime, then Run all once.')
    LLM, TOK = load_llm(MAIN_SPEC)
print('reasoner:', MAIN_SPEC['id'], MAIN_SPEC.get('quant'))

# retrieval embedder (bge-small); BM25 alone if it cannot be loaded
EMB = None
if not MOCK:
    try:
        from transformers import AutoModel, AutoTokenizer
        es = CFG['models']['embedder']
        EMB = (AutoTokenizer.from_pretrained(es['id']), AutoModel.from_pretrained(es['id']).to('cuda').eval())
    except Exception as e:
        log(f'embedder not available ({e.__class__.__name__}): retrieval uses BM25 only', 'WARN')
VERIFIER = None

[05:08:41] INFO  pip install transformers>=4.57.0 accelerate>=1.0 bitsandbytes>=0.45 rank_bm25
[05:09:00] WARN  No HF_TOKEN found. Public models still work; Gemma needs a token and an accepted licence.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5. Prompts, retrieval and the strategies

Every answer is JSON: `{"answer": letter, "evidence": [item IDs], "reasoning_steps": [...]}`. Invalid JSON gets one retry; if it fails again the answer counts as wrong (the rate is reported per run). The memory is described to the model as automatically produced and possibly wrong.

**Retrieval** (C, D): every memory item is scored against the question, its parse and sub-questions, and the options, by BM25 and bge-small (equal weights after normalising). Items are added in score order until the 600-token budget is full, then shown in time order. If the two best items are events with at most 8 events between them, those events are added too (temporal expansion).

**Verification** (D): the verifier sees the retrieved items, the question and the proposed answer with its cited IDs, and replies `{"supported": true|false}`. A symbolic check also requires every cited ID to exist among the shown items. If either fails, the items are retrieved again with twice the budget and the question is answered once more.

In [7]:
LET = 'ABCDE'
SYS = ('You answer multiple-choice questions about a video using only the video memory provided. The memory was '
       'produced automatically by computer vision and may contain errors or omissions. Reply with JSON only.')
BUDGET = CFG['llm']['retrieval_token_budget']
EXPAND = CFG['llm']['expansion_max_items']

def options_text(q):
    return '\n'.join(f'{LET[i]}. {o}' for i, o in enumerate(q['options']))

def parse_json(text):
    """The first JSON object (dict) in a reply, or None. Lists and other JSON values are ignored."""
    if not isinstance(text, str):
        return None
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    s = text.find('{')
    while s != -1:
        depth = 0
        for j in range(s, len(text)):
            depth += (text[j] == '{') - (text[j] == '}')
            if depth == 0:
                try:
                    v = json.loads(text[s:j + 1])
                    if isinstance(v, dict):
                        return v
                except json.JSONDecodeError:
                    pass
                break
        s = text.find('{', s + 1)
    return None

def letter_of(js, q):
    if not isinstance(js, dict):
        return None
    a = js.get('answer', '')
    if isinstance(a, list):
        a = a[0] if a else ''
    a = str(a).strip().upper()[:1]
    return a if a and a in LET[:len(q['options'])] else None

ANSWER_FMT = 'Reply as {"answer": "<letter>", "evidence": ["<IDs of the memory items you used, e.g. E3>"]}.'
ANSWER_FMT_STEPS = ('Reply as {"answer": "<letter>", "evidence": ["<IDs of the memory items you used>"], '
                    '"reasoning_steps": ["<at most 3 short steps>"]}.')

def mem_text_capped(m, cap_tokens=3000):
    """The full memory for strategy A/B, cut at about cap_tokens (pilot rule keeps prompts inside a T4)."""
    if m['tokens'] <= cap_tokens:
        return m['text'], False
    words = m['text'].split(' ')
    keep = int(len(words) * cap_tokens / m['tokens'])
    return ' '.join(words[:keep]) + '\n(... memory cut to fit)', True

def conv_answer(memtext, q, header='VIDEO MEMORY', extra='', steps=False):
    return [{'role': 'system', 'content': SYS},
            {'role': 'user', 'content': f'{header}:\n{memtext}\n\n{extra}QUESTION: {q["question"]}\nOPTIONS:\n{options_text(q)}\n\n'
                                        + (ANSWER_FMT_STEPS if steps else ANSWER_FMT)}]

def conv_parse(q):
    return [{'role': 'user', 'content':
             f'Analyse this question about a video. QUESTION: {q["question"]}\nOPTIONS:\n{options_text(q)}\n\n'
             'Reply with JSON only: {"type": "causal|temporal_before|temporal_after|temporal_during|descriptive", '
             '"target_object": "...", "target_actor": "...", "anchor_event": "...", "relation": "before|after|during|why|how|none", '
             '"sub_questions": ["2 to 4 short questions that, answered in order, answer the main question"]}'}]

def conv_verify(items_text, q, letter, evidence):
    opt = q['options'][LET.index(letter)] if letter else '(none)'
    return [{'role': 'user', 'content':
             f'MEMORY ITEMS:\n{items_text}\n\nQUESTION: {q["question"]}\nPROPOSED ANSWER: {opt}\n'
             f'CITED ITEMS: {", ".join(evidence) or "none"}\n\nDo the memory items support the proposed answer? '
             'Reply with JSON only: {"supported": true or false}'}]

# ---- retrieval ---------------------------------------------------------------------------------
_ITEM_TOK, _EMB_CACHE = {}, {}
def item_tokens(text):
    if text not in _ITEM_TOK:
        _ITEM_TOK[text] = len(TOK(text)['input_ids'])
    return _ITEM_TOK[text]

def embed(texts):
    import torch
    tk, md_ = EMB
    out = []
    for i in range(0, len(texts), 64):
        enc = tk(texts[i:i + 64], padding=True, truncation=True, max_length=256, return_tensors='pt').to('cuda')
        with torch.inference_mode():
            v = md_(**enc).last_hidden_state[:, 0]
        out.append(torch.nn.functional.normalize(v, dim=-1).float().cpu().numpy())
    return np.concatenate(out)

def retrieve(m, query, budget):
    items = m['items']
    if not items:
        return [], 0
    toks = [re.findall(r'[a-z0-9]+', it['text'].lower()) for it in items]
    qt = re.findall(r'[a-z0-9]+', query.lower())
    try:
        from rank_bm25 import BM25Okapi
        bm = np.array(BM25Okapi(toks).get_scores(qt))
    except Exception:
        bm = np.array([len(set(t) & set(qt)) for t in toks], float)
    score = (bm - bm.min()) / (bm.max() - bm.min() + 1e-9)
    if EMB is not None:
        ck = (m['format'], m['key'])
        if ck not in _EMB_CACHE:
            _EMB_CACHE[ck] = embed([it['text'] for it in items])
        sim = _EMB_CACHE[ck] @ embed([query])[0]
        score = 0.5 * score + 0.5 * (sim - sim.min()) / (sim.max() - sim.min() + 1e-9)
    order = list(np.argsort(-score))
    chosen, used = [], 0
    for k in order:
        t = item_tokens(items[k]['text'])
        if used + t > budget:
            continue
        chosen.append(k); used += t
    ev_idx = [k for k in order[:2] if items[k]['kind'] == 'event']
    if len(ev_idx) == 2:                                   # temporal expansion between the two best events
        a, b = sorted(ev_idx)
        between = [k for k in range(a + 1, b) if items[k]['kind'] == 'event' and k not in chosen]
        if 0 < len(between) <= EXPAND:
            extra = sum(item_tokens(items[k]['text']) for k in between)
            if used + extra <= budget * 1.25:
                chosen += between; used += extra
    chosen.sort(key=lambda k: (items[k]['t0'], k))
    return [items[k] for k in chosen], used

def query_of(q, parse):
    p = parse or {}
    parts = [q['question'], p.get('anchor_event') or '', p.get('target_object') or '', p.get('target_actor') or '',
             ' '.join(p.get('sub_questions') or []), ' '.join(q['options'])]
    return ' '.join(str(x) for x in parts if x)

# ---- strategies (all batched over a chunk of questions) ----------------------------------------
GEN_BATCH, GEN_BATCH_TOKENS = 8, 8000   # fits a T4; the learned batch size is kept between calls
def gen(convs, max_new, model=None, tok=None, temperature=0.0):
    return llm_generate(model or LLM, tok or TOK, convs, max_new, temperature,
                        batch_size=GEN_BATCH, max_batch_tokens=GEN_BATCH_TOKENS)

def answer_with_retry(convs, qs, recs, max_new, model=None, tok=None):
    outs, pl, gl = gen(convs, max_new, model, tok)
    for r, p, g in zip(recs, pl, gl):
        r['calls'] += 1; r['prompt_tokens'] += p; r['gen_tokens'] += g
    res = [(letter_of(parse_json(o), q), parse_json(o), o) for o, q in zip(outs, qs)]
    bad = [i for i, (L, js, o) in enumerate(res) if L is None]
    if bad:
        retry = [convs[i] + [{'role': 'assistant', 'content': res[i][2] or ''},
                             {'role': 'user', 'content': 'That was not valid. ' + ANSWER_FMT + ' JSON only.'}] for i in bad]
        outs2, pl2, gl2 = gen(retry, max_new, model, tok)
        for i, o, p, g in zip(bad, outs2, pl2, gl2):
            recs[i]['calls'] += 1; recs[i]['prompt_tokens'] += p; recs[i]['gen_tokens'] += g; recs[i]['retried'] = True
            js = parse_json(o)
            res[i] = (letter_of(js, qs[i]), js, o)
    return res

def do_parse(qs, recs):
    outs, pl, gl = gen([conv_parse(q) for q in qs], 128)
    parses = []
    for r, o, p, g in zip(recs, outs, pl, gl):
        r['calls'] += 1; r['prompt_tokens'] += p; r['gen_tokens'] += g
        js = parse_json(o)
        parses.append(js if isinstance(js, dict) else {})
        r['parse'] = parses[-1]
    return parses

def finish(r, q, L, js, raw):
    r.update(pred=L, gold=LET[q['gold']], correct=bool(L == LET[q['gold']]), valid_json=L is not None,
             evidence=[str(x) for x in ((js or {}).get('evidence') or [])][:10] if isinstance(js, dict) else [],
             steps=(js or {}).get('reasoning_steps') if isinstance(js, dict) else None,
             raw=None if L is not None else (raw or '')[:300])

def run_chunk(run, qs):
    strat = run['strategy']
    mems = [memory(run['format'], q['key']) for q in qs]
    recs = [{'uid': q['uid'], 'run': run['id'], 'qset': run['qset'], 'format': run['format'], 'strategy': strat,
             'key': q['key'], 'type': q['type'], 'model': MAIN_SPEC['id'], 'calls': 0, 'prompt_tokens': 0,
             'gen_tokens': 0, 'mem_tokens': m['tokens']} for q, m in zip(qs, mems)]
    if strat in ('A', 'A-SC'):
        texts = [mem_text_capped(m) for m in mems]
        for r, (_, cut) in zip(recs, texts):
            r['memory_cut'] = cut
        convs = [conv_answer(t, q) for (t, _), q in zip(texts, qs)]
        if strat == 'A':
            for r, q, (L, js, raw) in zip(recs, qs, answer_with_retry(convs, qs, recs, 64)):
                finish(r, q, L, js, raw)
        else:
            n = CFG['llm']['self_consistency_samples']
            outs, pl, gl = gen([c for c in convs for _ in range(n)], 64,
                               temperature=CFG['llm']['self_consistency_temperature'])
            for i, (r, q) in enumerate(zip(recs, qs)):
                votes = [letter_of(parse_json(o), q) for o in outs[i * n:(i + 1) * n]]
                r['calls'] += n; r['prompt_tokens'] += sum(pl[i * n:(i + 1) * n]); r['gen_tokens'] += sum(gl[i * n:(i + 1) * n])
                valid = [v for v in votes if v]
                L = max(valid, key=valid.count) if valid else None     # ties: earliest sample
                finish(r, q, L, parse_json(outs[i * n]), outs[i * n])
                r['votes'] = votes
        return recs
    parses = do_parse(qs, recs)
    if strat == 'B':
        texts = [mem_text_capped(m) for m in mems]
        convs = [conv_answer(t, q, extra='SUB-QUESTIONS (answer them in order, then the main question):\n'
                             + '\n'.join(f'- {s}' for s in (p.get('sub_questions') or [])[:4]) + '\n\n', steps=True)
                 for (t, _), q, p in zip(texts, qs, parses)]
        for r, q, (L, js, raw) in zip(recs, qs, answer_with_retry(convs, qs, recs, 160)):
            finish(r, q, L, js, raw)
        return recs
    # C, D, D-X: retrieval
    shown = []
    for r, m, q, p in zip(recs, mems, qs, parses):
        items, used = retrieve(m, query_of(q, p), BUDGET)
        shown.append(items)
        r['retrieved'] = [it['id'] for it in items]; r['retrieved_tokens'] = used
    convs = [conv_answer('\n'.join(it['text'] for it in its) or '(no items)', q, header='RETRIEVED MEMORY ITEMS', steps=True)
             for its, q in zip(shown, qs)]
    res = answer_with_retry(convs, qs, recs, 128)
    if strat == 'C':
        for r, q, (L, js, raw) in zip(recs, qs, res):
            finish(r, q, L, js, raw)
        return recs
    # verification
    vm, vt = (VERIFIER if strat == 'D-X' else (LLM, TOK))
    vconvs = [conv_verify('\n'.join(it['text'] for it in its), q, L, [str(e) for e in ((js or {}).get('evidence') or [])]
                          if isinstance(js, dict) else [])
              for its, q, (L, js, raw) in zip(shown, qs, res)]
    vout, vpl, vgl = gen(vconvs, 24, vm, vt)
    redo = []
    for i, (r, its, (L, js, raw), o, p, g) in enumerate(zip(recs, shown, res, vout, vpl, vgl)):
        r['calls'] += 1; r['prompt_tokens'] += p; r['gen_tokens'] += g
        vj = parse_json(o)
        supported = bool(isinstance(vj, dict) and vj.get('supported') is True)
        cited = [str(e) for e in ((js or {}).get('evidence') or [])] if isinstance(js, dict) else []
        ids_ok = bool(cited) and all(c in {it['id'] for it in its} for c in cited)
        r.update(verifier_supported=supported, symbolic_ok=ids_ok, verifier_model=(VERIFIER_ID if strat == 'D-X' else MAIN_SPEC['id']))
        if not (supported and ids_ok and L):
            redo.append(i)
    final = list(res)
    if redo:
        items2 = []
        for i in redo:
            its, used = retrieve(mems[i], query_of(qs[i], parses[i]), BUDGET * 2)
            items2.append(its); recs[i]['retrieved_2'] = [it['id'] for it in its]
        convs2 = [conv_answer('\n'.join(it['text'] for it in its) or '(no items)', qs[i], header='RETRIEVED MEMORY ITEMS', steps=True)
                  for its, i in zip(items2, redo)]
        sub_recs = [recs[i] for i in redo]
        for i, out in zip(redo, answer_with_retry(convs2, [qs[i] for i in redo], sub_recs, 128)):
            recs[i]['reanswered'] = True
            final[i] = out if out[0] else final[i]       # keep the first answer if the second is invalid
    for r, q, (L, js, raw) in zip(recs, qs, final):
        finish(r, q, L, js, raw)
    return recs

## 6. Run

Runs are processed in priority order; within each run, this person's questions are answered in chunks of 48. A question whose memory is missing (its video had no CV result) is skipped and listed. The cell stops cleanly after 110 minutes; run it again to continue.

In [8]:
CHUNK = 48                 # questions per step; larger chunks let each batch fill up
VERIFIER_ID = None
RUN_LOG = Run(params={'step': 'llm_qa', 'shard': SHARD, 'n_shards': N_SHARDS, 'memories': MEM_PACK,
                      'model': MAIN_SPEC['id'], 'runs': len(RUNS), 'limit': LIMIT_QUESTIONS})
budget_s = CFG['runtime']['chunk_minutes'] * 60
t_start, last_sync, stop = time.time(), time.time(), False
missing_mem = {}
try:
    for run in RUNS:
        qs_all = [q for q in QSETS[run['qset']] if q['shard'] == SHARD]
        if LIMIT_QUESTIONS:
            qs_all = qs_all[:LIMIT_QUESTIONS]
        have = [q for q in qs_all if (MEM_LOCAL / run['format'] / f"{q['key']}.json").exists()]
        if len(have) < len(qs_all):
            missing_mem[run['id']] = len(qs_all) - len(have)
        done = answered(run['id'])
        todo = [q for q in have if q['uid'] not in done]
        if not todo:
            continue
        if run['strategy'] == 'D-X' and VERIFIER is None:
            alt = CFG['models']['reasoner_alt'] if HAS_TOKEN else CFG['models']['reasoner_small']
            try:
                VERIFIER = load_llm(alt); VERIFIER_ID = alt['id']
            except Exception as e:
                alt = CFG['models']['reasoner_small']
                log(f'verifier {CFG["models"]["reasoner_alt"]["id"]} not available ({e.__class__.__name__}): using {alt["id"]}', 'WARN')
                VERIFIER = load_llm(alt); VERIFIER_ID = alt['id']
            log(f'D-X verifier: {VERIFIER_ID}')
        log(f'{run["id"]}: {len(todo)} questions to do ({len(done)} already answered)')
        t_run = time.time()
        for c in range(0, len(todo), CHUNK):
            if time.time() - t_start > budget_s:
                log('time budget reached: run this cell again to continue', 'WARN'); stop = True; break
            chunk = todo[c:c + CHUNK]
            t0 = time.time()
            recs = run_chunk(run, chunk)
            per = (time.time() - t0) / len(chunk)
            for r in recs:
                r['seconds'] = round(per, 2)
            out = LOCAL_PRED / run['id'] / f's{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S-%f}.jsonl'
            out.parent.mkdir(parents=True, exist_ok=True)
            for r in recs:
                append_jsonl(out, r)
            if time.time() - last_sync > SYNC_EVERY_MIN * 60:
                sync_preds(); last_sync = time.time()
        n_done = len(answered(run['id']) & {q['uid'] for q in have})
        acc = np.mean([r['correct'] for f in (LOCAL_PRED / run['id']).glob(f's{SHARD}_*.jsonl') for r in read_jsonl(f)])
        log(f'{run["id"]}: {n_done}/{len(have)} answered | {(time.time() - t_run) / max(1, len(todo)):.1f} s per question '
            f'| accuracy so far (this person) {acc:.3f}')
        if stop:
            break
finally:
    sync_preds()
if missing_mem:
    log(f'questions skipped because their memory is missing: {missing_mem}', 'WARN')

[05:18:55] INFO  run started: 05_llm_qa-20261019-051855-4e0485
[05:18:55] INFO  main.M1.A: 93 questions to do (100 already answered)
[05:27:56] INFO  uploaded 05_shard1_20261019-052756_00001.zip: 1 files, verified on Drive
[05:33:39] INFO  main.M1.A: 93/93 answered | 9.5 s per question | accuracy so far (this person) 0.366
[05:33:39] INFO  star.M1.A: 96 questions to do (94 already answered)
[05:36:25] INFO  uploaded 05_shard1_20261019-053624_00002.zip: 2 files, verified on Drive
[05:38:24] INFO  star.M1.A: 96/96 answered | 3.0 s per question | accuracy so far (this person) 0.562
[05:38:24] INFO  uploaded 05_shard1_20261019-053824_00001.zip: 1 files, verified on Drive


## 7. Progress and accuracy so far

Covers every person's answers that are on Drive (restored at the start of this session) plus this session's. Accuracies are **not results yet**: the final numbers, confidence intervals and tests come from notebooks 08 and 09 once all four people have finished.

In [9]:
try:
    sync_preds()
except DriveSyncError as e:
    log(str(e), 'ERROR')
rows = []
for run in RUNS:
    recs = [r for f in (LOCAL_PRED / run['id']).glob('*.jsonl') for r in read_jsonl(f)]
    n_total = len(QSETS[run['qset']]) if not LIMIT_QUESTIONS else None
    mine = [r for r in recs if shard_of(r['uid'].split('__')[0]) == SHARD]
    n_mine = sum(q['shard'] == SHARD for q in QSETS[run['qset']])
    rows.append({'run': run['id'], 'answered_all': len({r['uid'] for r in recs}), 'of': n_total,
                 'mine': f"{len({r['uid'] for r in mine})}/{n_mine}",
                 'accuracy': round(np.mean([r['correct'] for r in recs]), 3) if recs else None,
                 'valid_json': round(np.mean([r['valid_json'] for r in recs]), 3) if recs else None,
                 'calls_per_q': round(np.mean([r['calls'] for r in recs]), 2) if recs else None,
                 's_per_q': round(np.mean([r['seconds'] for r in recs]), 2) if recs else None})
T = pd.DataFrame(rows)
print(T.to_string(index=False))
left_mine = sum(int(x.split('/')[1]) - int(x.split('/')[0]) for x in T['mine'])
default_s = {'A': 9.0, 'A-SC': 25.0, 'B': 12.0, 'C': 12.0, 'D': 18.0, 'D-X': 20.0}   # measured on a free T4
est = sum((int(x.split('/')[1]) - int(x.split('/')[0])) * (sp if sp == sp and sp else default_s[r.split('.')[-1]])
          for r, x, sp in zip(T['run'], T['mine'], T['s_per_q'])) / 3600
print(f'\nthis person: {left_mine} question-runs left, roughly {est:.1f} GPU hours at the speeds measured so far')
save_json({'shard': SHARD, 'left': left_mine, 'est_hours': round(est, 2), 'table': rows,
           'updated': dt.datetime.now().isoformat(timespec='seconds')}, P['pred'] / f'progress_05_shard{SHARD}.json')
RUN_LOG.finish('done' if left_mine == 0 else 'partial', notes=f'{left_mine} question-runs left')

      run  answered_all  of  mine  accuracy  valid_json  calls_per_q  s_per_q
main.M1.A           193 400 93/93     0.420       0.813         1.20     9.18
star.M1.A           190 360 96/96     0.505       0.979         1.03     2.69

this person: 0 question-runs left, roughly 0.0 GPU hours at the speeds measured so far
[05:38:26] INFO  run finished: 05_llm_qa-20261019-051855-4e0485 [done] 19.5 min, peak RAM 1.83 GB, peak VRAM 12.99 GB


## 8. Free the Colab disk (optional)

Only when finished with GPU work in this session: set `FREE_DISK_NOW = True` and run this cell. It uploads any remaining answers first, then deletes the downloaded model weights. Off by default so that Run all never deletes the model in use.

In [10]:
FREE_DISK_NOW = False
if FREE_DISK_NOW:
    sync_preds()
    for n in ['LLM', 'TOK', 'VERIFIER', 'EMB']:
        globals().pop(n, None)
    free_memory()
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    shutil.rmtree(hub, ignore_errors=True)
print('local disk free:', _gb(shutil.disk_usage(LOCAL).free), 'GB')

local disk free: 54.57 GB


## What to send back

The table from section 7 after the first session, so the speed per question can be checked and the remaining time estimated.